## LOGISTIC REGRESSION

In [ ]:
import os
os.chdir('../..')
print(os.getcwd())
import pandas as pd
import numpy as np
from benchmarks_august.targets.logreg import logreg
from sklearn.model_selection import train_test_split

df = pd.read_csv("benchmarks_august/datasets/diabetes.csv")
X = df.drop("Outcome", axis=1).values.astype(float)
y = df["Outcome"].values.astype(float)

# Handle implicit missingness: zero means missing in these columns
for col_idx in [1, 2, 3, 4, 5]:  # Glucose, BP, Skin, Insulin, BMI
    mask = X[:, col_idx] == 0
    X[mask, col_idx] = np.nan
X = np.where(np.isnan(X), np.nanmean(X, axis=0), X)  # mean impute

# Standardize
X = (X - X.mean(axis=0)) / X.std(axis=0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

target = logreg(X_train, y_train, prior={"kind": "gaussian", "scale": 1.0})

In [ ]:
from benchmarks_august.samplers.warmstart import warmup_reference
from benchmarks_august.samplers import build_sampler, build_kappa, apply_preprocess

N = 10000

# --- Boomerang ---
boom = build_sampler("boomerang", target, N=N, refresh_rate=1.0, t_max = 1.0)
warmup_reference(boom, n_rounds=3, n_pilot=500, tune_refresh=True)
boom.reset(N=N)
boom.sample_auto(diagnostics=True)

# --- Boomerang PLI ---
boom_pli = build_sampler("boomerang_pli", target, N=N, refresh_rate=1.0)
warmup_reference(boom_pli, n_rounds=3, n_pilot=500, tune_refresh=True)
boom_pli.reset(N=N)
boom_pli.sample_auto(diagnostics=True)

# # --- Factorized Boomerang ---
# fact_boom = build_sampler("factorized_boomerang", target, N=N, refresh_rate=1.0, t_max = 1.0)
# apply_preprocess(fact_boom, target, {"method": "diagonal"})
# fact_boom.sample_auto(diagnostics=True)

# --- Sticky Boomerang ---
kappa = build_kappa({"kind": "uniform", "gamma_prior": 0.5}, target)
sticky = build_sampler("sticky_boomerang", target, N=N, kappa=kappa, refresh_rate=1.0, t_max = 1.0)
warmup_reference(sticky, n_rounds=3, n_pilot=500, tune_refresh=True)
sticky.reset(N=N)
sticky.sample_auto(diagnostics=True)

# --- Sticky Boomeran PLI ---
kappa = build_kappa({"kind": "uniform", "gamma_prior": 0.5}, target)
sticky_pli = build_sampler("sticky_boomerang_pli", target, N=N, kappa=kappa, refresh_rate=1.0)
warmup_reference(sticky_pli, n_rounds=3, n_pilot=500, tune_refresh=True)
sticky_pli.reset(N=N)
sticky_pli.sample_auto(diagnostics=True)

In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(C=1.0, penalty='l2', fit_intercept=False).fit(X_train, y_train)
print("sklearn:", lr.coef_[0])

In [ ]:
import numpy as np

print("Boomerang:", boom.Position.mean(axis=0))
print("Boomerang PLI:", boom_pli.Position.mean(axis=0))
# print("Factorized Boomerang:", fact_boom.Position.mean(axis=0))
print("Sticky:   ", sticky.Position.mean(axis=0))
print("Sticky PLI:   ", sticky_pli.Position.mean(axis=0))
# print("Sticky zero fraction:    ", (sticky.Position == 0).mean(axis=0))
# print("Sticky PLI zero fraction:    ", (sticky_pli.Position == 0).mean(axis=0))

In [ ]:
from sazz.samplers.boomerang_sampler.utils import resample_pdmp_path, resample_sticky_pdmp_path
t, x = resample_pdmp_path(boom, n_samples=10000)
t_pli, x_pli = resample_pdmp_path(boom_pli, n_samples=10000)
# t_fact, x_fact = resample_pdmp_path(fact_boom, n_samples=10000)
t_sticky, x_sticky = resample_sticky_pdmp_path(sticky, n_samples=10000)
t_sticky_pli, x_sticky_pli = resample_sticky_pdmp_path(sticky_pli, n_samples=10000)

In [ ]:
import time
import pymc as pm
start = time.perf_counter()
with pm.Model() as logreg_model:
    beta = pm.Normal("beta", mu=0, sigma=1, shape=X_train.shape[1])
    logits = pm.math.dot(X_train, beta)
    y_obs = pm.Bernoulli("y", logit_p=logits, observed=y_train)
    trace = pm.sample(3000, tune=2000, cores=1, random_seed=42)
nuts_wall = time.perf_counter() - start

nuts_means = trace.posterior["beta"].mean(dim=["chain", "draw"]).values
nuts_samples = trace.posterior["beta"].values.reshape(-1, X_train.shape[1])

In [ ]:
from benchmarks_august.analysis.metrics import sample_quality, sampler_efficiency, logreg_performance, sticky_ess

# --- 1. Sample quality ---
# sample_quality(x, sklearn_coefs=lr.coef_[0], label="Boomerang")
# sample_quality(x_pli, sklearn_coefs=lr.coef_[0], label="Boomerang PLI")
# sample_quality(nuts_samples, sklearn_coefs=lr.coef_[0], label="NUTS")
# # sample_quality(x_fact, sklearn_coefs=lr.coef_[0], label="Factorized Boomerang")
# sample_quality(x_sticky, sklearn_coefs=lr.coef_[0], label="Sticky")
# sample_quality(x_sticky_pli, sklearn_coefs=lr.coef_[0], label="Sticky PLI")

In [ ]:
sampler_efficiency({
    "Boomerang": boom,
    "Boomerang PLI": boom_pli,
    "Sticky": sticky,
    "Sticky PLI": sticky_pli,
})

# Extract NUTS diagnostics
nuts_stats = trace.sample_stats
n_samples = 3000
n_tune = 2000

from benchmarks_august.analysis.metrics import _ess_batch_means
from sklearn.decomposition import PCA

pca = PCA(n_components=1).fit(nuts_samples)
nuts_ess = _ess_batch_means(nuts_samples @ pca.components_[0])

name = "NUTS"
print(f"{name:<23s}        {nuts_wall:.1f}")

In [ ]:
# --- 2. Model performance ---
logreg_performance(X_train, y_train, X_test, y_test, {
    "sklearn MAP": lr.coef_[0].reshape(1, -1),
    "Boomerang": x,
    "Boomerang PLI": x_pli,
    "NUTS": nuts_samples,
    # "Factorized Boomerang": x_fact,
    "Sticky": x_sticky,
    "Sticky PLI": x_sticky_pli,
})

In [ ]:
# Sticky-aware ESS (standard ESS is misleading for sticky samples)
print("Sticky — ESS diagnostics:")
s = sticky_ess(x_sticky, burnin_frac=0)
print(f"  Mean inclusion ESS:  {s['mean_inclusion_ess']:.0f}")
print(f"  Model size ESS:      {s['model_size_ess']:.0f}")
print(f"  Mean active ESS:     {s['mean_active_ess']:.0f}")

print("Sticky PLI — ESS diagnostics:")
s = sticky_ess(x_sticky_pli, burnin_frac=0)
print(f"  Mean inclusion ESS:  {s['mean_inclusion_ess']:.0f}")
print(f"  Model size ESS:      {s['model_size_ess']:.0f}")
print(f"  Mean active ESS:     {s['mean_active_ess']:.0f}")

## Neural Network

In [ ]:
from benchmarks_august.targets.bnn_classification import bnn_classification
from benchmarks_august.samplers.warmstart.bnn_classification_warmstart import adam_fisher
from benchmarks_august.samplers import build_sampler, apply_preprocess

H=10
layers = [X.shape[1], H, 1]

# 1. Build target (E and gradE come from here)
target = bnn_classification(X_train, y_train, layer_sizes=layers,
    prior={"kind": "layered_gaussian",
           "sigma_w_layers": [1.0, 1.0],
           "sigma_b_layers": [1.0, 1.0]})

# 2. Warmstart (sets x_ref and Sigma_inv)
ws = adam_fisher(target, n_epochs=3000, lr=3e-3, l2_weight=0.001, l1_weight=0.02)
target.x_ref = ws["x_ref"]
target.Sigma_inv = ws["Sigma_inv"]

# 3. Build kappa from target.meta
w_prior = 0.5
kappa = np.empty(target.D)
kappa[~target.meta["weight_mask"]] = 1e6
for l, (w_sl, b_sl) in enumerate(target.meta["slices"]):
    sigma_l = target.meta["sigma_w_layers"][l]
    kappa[w_sl] = (1 - w_prior) / w_prior / (sigma_l * np.sqrt(2 * np.pi))

In [ ]:
from benchmarks_august.samplers.warmstart import warmup_reference
N=10000
sampler = build_sampler("boomerang_pli", target, N=N,
                        refresh_rate=1.0)
warmup_reference(sampler, n_rounds=3, n_pilot=500, tune_refresh=True)
sampler.reset(N=N)
sampler.sample_auto()

sampler_raw = build_sampler("boomerang_pli", target, N=N,
                        refresh_rate=1.0)
apply_preprocess(sampler_raw, target, {"method": "diagonal"})
sampler_raw.sample_auto()

In [ ]:
sampler_sticky = build_sampler("sticky_boomerang_pli", target, N=N,
                        kappa=kappa, refresh_rate=1.0, cold_start_threshold=0.05)
warmup_reference(sampler_sticky, n_rounds=3, n_pilot=500, tune_refresh=True)
sampler_sticky.reset(N=N)
sampler_sticky.sample_auto()

sampler_sticky_raw = build_sampler("sticky_boomerang_pli", target, N=N,
                        kappa=kappa, refresh_rate=1.0, cold_start_threshold=0.05)
apply_preprocess(sampler_sticky_raw, target, {"method": "diagonal"})
sampler_sticky_raw.sample_auto()

In [ ]:
from sazz.samplers.boomerang_sampler.utils import resample_pdmp_path, resample_sticky_pdmp_path
t, x = resample_pdmp_path(sampler, n_samples=50000)
t_sticky, x_sticky = resample_sticky_pdmp_path(sampler_sticky, n_samples=50000)

In [ ]:
from benchmarks_august.analysis.metrics import bnn_performance

bnn_performance(X_train, y_train, X_test, y_test,
    {
        "Adam MAP": target.x_ref.reshape(1, -1),
        # "Boomerang raw": x_raw,
        "Boomerang adaptive": x,
        # "Sticky raw": x_sticky_raw,
        "Sticky adaptive": x_sticky,
    },
    shapes=target.meta["shapes"],
    slices=target.meta["slices"],
    weight_mask=target.meta["weight_mask"])